# Sentinel — Data Deep-Dive 🔬

A visual tour of **every dataset** behind the predictive-maintenance system: what each table is,
how they connect, which signals actually matter, and the data quirks you must know.

> 100 machines · 1 year · hourly. Five tables — four are **inputs**, one is the **answer key**.


## 0 · Setup & house style

In [ ]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if not (ROOT/"data"/"raw").exists() and (ROOT.parent/"data"/"raw").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); RAW = ROOT/"data"/"raw"

# ---- house style: clean, modern ----
INK, MUTE, GRID = "#1e293b", "#64748b", "#e2e8f0"
PALETTE = {"volt":"#6366f1","rotate":"#0ea5e9","pressure":"#10b981","vibration":"#f43f5e",
           "indigo":"#6366f1","sky":"#0ea5e9","emerald":"#10b981","rose":"#f43f5e",
           "amber":"#f59e0b","violet":"#8b5cf6","slate":"#94a3b8"}
mpl.rcParams.update({
    "figure.facecolor":"white","axes.facecolor":"white","axes.edgecolor":GRID,
    "axes.grid":True,"grid.color":GRID,"grid.alpha":0.6,"axes.spines.top":False,
    "axes.spines.right":False,"axes.titlesize":12,"axes.titleweight":"bold",
    "axes.labelcolor":MUTE,"xtick.color":MUTE,"ytick.color":MUTE,"text.color":INK,
    "font.size":10,"axes.titlecolor":INK,
})
SENS = ["volt","rotate","pressure","vibration"]
COMPS = ["comp1","comp2","comp3","comp4"]
print("root:", ROOT)


---
## 1 · The five datasets — a catalog

| Table | Role | Grain (one row =) | Key columns |
|---|---|---|---|
| **telemetry** | 🟦 main input | one machine × one hour | volt, rotate, pressure, vibration |
| **errors** | 🟦 main input | a non-fatal warning | errorID (error1–5) |
| **maintenance** | 🟦 main input | a component replacement | comp (comp1–4) |
| **machines** | ⬜ static context | one machine | model (Type 1–4), age |
| **failures** | 🟥 **answer key** | an actual breakdown | failure (comp1–4) |

**The golden rule:** `failures` is used *only* to create labels and fit survival models —
**never** as a model input. Everything the model sees comes from the four blue tables.


In [ ]:
tele = pd.read_csv(RAW/"PdM_telemetry.csv", parse_dates=["datetime"])
errs = pd.read_csv(RAW/"PdM_errors.csv", parse_dates=["datetime"])
maint = pd.read_csv(RAW/"PdM_maint.csv", parse_dates=["datetime"])
fails = pd.read_csv(RAW/"PdM_failures.csv", parse_dates=["datetime"])
machines = pd.read_csv(RAW/"PdM_machines.csv")
for nm, d in [("telemetry",tele),("errors",errs),("maintenance",maint),("failures",fails),("machines",machines)]:
    print(f"{nm:12} {d.shape[0]:>8,} rows  x {d.shape[1]} cols")


### 1.1 · How the tables connect — schema map

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.6)); ax.axis("off")
ax.set_xlim(0,10); ax.set_ylim(0,7)

def box(x, y, w, h, title, cols, color):
    ax.add_patch(FancyBboxPatch((x,y), w, h, boxstyle="round,pad=0.04,rounding_size=0.12",
                 fc=color, ec="none", alpha=0.16))
    ax.add_patch(FancyBboxPatch((x,y+h-0.55), w, 0.55, boxstyle="round,pad=0.02,rounding_size=0.1",
                 fc=color, ec="none", alpha=0.95))
    ax.text(x+w/2, y+h-0.28, title, ha="center", va="center", color="white", fontweight="bold", fontsize=11)
    ax.text(x+0.15, y+h-0.8, cols, ha="left", va="top", color=INK, fontsize=8.4)

box(0.3, 3.9, 3.0, 2.8, "TELEMETRY 🟦", "datetime\nmachineID  (FK)\nvolt\nrotate\npressure\nvibration", PALETTE["indigo"])
box(3.6, 4.6, 2.7, 2.1, "ERRORS 🟦", "datetime\nmachineID  (FK)\nerrorID", PALETTE["sky"])
box(6.6, 4.6, 3.1, 2.1, "MAINTENANCE 🟦", "datetime\nmachineID  (FK)\ncomp", PALETTE["emerald"])
box(3.6, 0.3, 2.7, 2.0, "MACHINES ⬜", "machineID  (PK)\nmodel (Type)\nage", PALETTE["slate"])
box(6.6, 0.3, 3.1, 2.0, "FAILURES 🟥", "datetime\nmachineID  (FK)\nfailure  (LABEL)", PALETTE["rose"])

ax.text(5.0, 3.35, "join key:  machineID  (+ datetime for time-series)", ha="center",
        fontsize=10, color=MUTE, style="italic",
        bbox=dict(boxstyle="round,pad=0.4", fc="#f1f5f9", ec=GRID))
for (x,y) in [(3.6,5.6),(6.6,5.6)]:
    ax.add_patch(FancyArrowPatch((3.3,5.3),(x,y), arrowstyle="-", color=PALETTE["slate"], lw=1.2, alpha=0.6))
ax.add_patch(FancyArrowPatch((5.0,2.3),(1.8,3.9), arrowstyle="-|>", color=PALETTE["slate"], lw=1.2, alpha=0.6))
ax.set_title("Entity map — 4 blue inputs + static context, joined on machineID; red = label", loc="center", fontsize=12)
plt.tight_layout(); plt.show()


### 1.2 · Sample rows (`.head()`) + column types for every table

In [ ]:
try:
    from IPython.display import display, Markdown
except Exception:
    def Markdown(x): return x   # headless fallback
TABLES = [("telemetry",tele),("errors",errs),("maintenance",maint),("failures",fails),("machines",machines)]
for nm, d in TABLES:
    display(Markdown(f"**`{nm}`** — {len(d):,} rows × {d.shape[1]} cols"))
    display(d.head())


In [ ]:
# Column types + non-null counts for each table (data description)
for nm, d in TABLES:
    info = pd.DataFrame({
        "dtype": d.dtypes.astype(str),
        "non_null": d.notna().sum(),
        "nulls": d.isna().sum(),
        "n_unique": d.nunique(),
        "example": [d[c].dropna().iloc[0] if d[c].notna().any() else None for c in d.columns],
    })
    display(Markdown(f"**`{nm}`** column description"))
    display(info)


In [ ]:
# Numeric summary (.describe) for the sensor telemetry — the main numeric source
display(Markdown("**Telemetry — numeric summary** (`.describe()`)"))
display(tele[SENS].describe().T.round(2))
display(Markdown("**Machines — age summary**"))
display(machines[["age"]].describe().T.round(1))


---
## 2 · Telemetry — the primary signal 🟦

876,000 hourly readings of four sensors. This is the bulk of the data and the richest source.


In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(14, 6.5))
for a, s in zip(ax.ravel(), SENS):
    a.hist(tele[s], bins=70, color=PALETTE[s], alpha=0.85, edgecolor="white", linewidth=0.3)
    a.axvline(tele[s].mean(), color=INK, ls="--", lw=1, alpha=0.6)
    a.set_title(f"{s}   (μ={tele[s].mean():.1f}, σ={tele[s].std():.1f})")
fig.suptitle("Sensor distributions — each roughly bell-shaped around a nominal operating point",
             fontsize=13, fontweight="bold"); plt.tight_layout(); plt.show()


In [ ]:
# Sensor correlation + a sample machine's week
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={"width_ratios":[1,2]})
corr = tele[SENS].corr()
im = ax[0].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax[0].set_xticks(range(4)); ax[0].set_xticklabels(SENS, rotation=45, ha="right")
ax[0].set_yticks(range(4)); ax[0].set_yticklabels(SENS); ax[0].grid(False)
for (i,j),v in np.ndenumerate(corr.values): ax[0].text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=9)
ax[0].set_title("Sensor correlation (near-independent)"); fig.colorbar(im, ax=ax[0], fraction=0.046)

m = tele[tele.machineID==1].set_index("datetime").loc["2015-03-01":"2015-03-08"]
for s in SENS:
    ax[1].plot(m.index, (m[s]-m[s].mean())/m[s].std(), color=PALETTE[s], lw=1, label=s, alpha=0.85)
ax[1].set_title("Machine 1 · one week · sensors (z-scored)"); ax[1].legend(ncol=4, fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
# Do machine TYPES differ? mean sensor per model
tm = tele.merge(machines[["machineID","model"]], on="machineID")
means = tm.groupby("model")[SENS].mean()
means.plot.bar(figsize=(11,3.6), color=[PALETTE[s] for s in SENS], width=0.8)
plt.title("Mean sensor reading by machine Type — types have different operating points")
plt.xticks(rotation=0); plt.ylabel("mean"); plt.legend(ncol=4, fontsize=8); plt.tight_layout(); plt.show()


---
## 3 · Errors 🟦 — discrete warning events

Non-fatal codes (error1–5). Sparse but often **precede failures** — a key leading indicator.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={"width_ratios":[1,2]})
errs["errorID"].value_counts().sort_index().plot.bar(ax=ax[0], color=PALETTE["amber"], width=0.7)
ax[0].set_title("Errors per type"); ax[0].tick_params(axis="x", rotation=0)

# machine x error heatmap (who errors, and which codes)
piv = errs.assign(n=1).pivot_table(index="machineID", columns="errorID", values="n", aggfunc="sum", fill_value=0)
im = ax[1].imshow(piv.T, aspect="auto", cmap="YlOrRd")
ax[1].set_yticks(range(piv.shape[1])); ax[1].set_yticklabels(piv.columns)
ax[1].set_xlabel("machineID"); ax[1].set_title("Error frequency heatmap (error type × machine)")
ax[1].grid(False); fig.colorbar(im, ax=ax[1], fraction=0.025)
plt.tight_layout(); plt.show()


---
## 4 · Maintenance 🟦 — component replacements

When each of the 4 components was replaced. Drives the **part-age** features and the survival cycle.


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
maint["comp"].value_counts().sort_index().plot.bar(ax=ax[0], color=PALETTE["emerald"], width=0.7)
ax[0].set_title("Replacements per component"); ax[0].tick_params(axis="x", rotation=0)

# interval between replacements per component (box plot)
gaps = {c: [] for c in COMPS}
for (mid,c), g in maint.sort_values("datetime").groupby(["machineID","comp"]):
    d = g["datetime"].values
    gaps[c] += [(d[i+1]-d[i])/np.timedelta64(1,"D") for i in range(len(d)-1)]
ax[1].boxplot([gaps[c] for c in COMPS], tick_labels=COMPS, showfliers=False,
              patch_artist=True, boxprops=dict(facecolor=PALETTE["emerald"], alpha=0.4))
ax[1].set_title("Days between replacements"); ax[1].set_ylabel("days")

# THE SEED ARTIFACT: histogram of maintenance dates
maint.set_index("datetime").resample("W").size().plot(ax=ax[2], color=PALETTE["violet"])
ax[2].set_title("Replacements over time\n(note the 2014-06 seed spike)")
plt.tight_layout(); plt.show()
print("NOTE - Data quirk: every component is 'installed' on 2014-06-01 (a seed), 6 months before telemetry starts.")
print("  So at the start of 2015 many parts look very old — real, but inflates early 'overdue' counts.")


---
## 5 · Failures 🟥 — the answer key (what we predict)

Only **761** across the whole year — the rare positive class. Everything downstream is shaped by
this rarity.


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
fails["failure"].value_counts().sort_index().plot.bar(ax=ax[0], color=PALETTE["rose"], width=0.7)
ax[0].set_title("Failures per component"); ax[0].tick_params(axis="x", rotation=0)
fm = fails.merge(machines[["machineID","model"]], on="machineID")
fm["model"].value_counts().sort_index().plot.bar(ax=ax[1], color=PALETTE["indigo"], width=0.7)
ax[1].set_title("Failures per machine Type"); ax[1].tick_params(axis="x", rotation=0)
fails.set_index("datetime").resample("ME").size().plot(ax=ax[2], color=PALETTE["rose"], marker="o")
ax[2].set_title("Failures per month")
plt.tight_layout(); plt.show()


In [ ]:
# Class imbalance — the defining property of the data
n_pos = len(fails); n_all = len(tele)
fig, ax = plt.subplots(figsize=(7,2.6))
ax.barh(["machine-hours"], [n_all], color=PALETTE["slate"], alpha=0.4, label=f"healthy ({n_all:,})")
ax.barh(["machine-hours"], [n_pos*100], color=PALETTE["rose"], label=f"failure-linked (×100 to see it)")
ax.set_title(f"Class imbalance — failures are {n_pos/n_all*100:.3f}% of all hours"); ax.legend(); ax.grid(False)
plt.tight_layout(); plt.show()


---
## 6 · Which data actually matters? — the failure signature 🔍

We align every failure to hour 0 and average the sensors/errors in the **24 hours before** it.
Signals that deviate from normal *before* failure are the **leading indicators** the model exploits.


In [ ]:
# Build per-machine telemetry index once
tele_by_m = {mid: g.set_index("datetime").sort_index() for mid, g in tele.groupby("machineID")}
norm = {s: (tele[s].mean(), tele[s].std()) for s in SENS}

rows = []
for _, f in fails.iterrows():
    g = tele_by_m.get(f["machineID"])
    if g is None: continue
    w = g.loc[f["datetime"]-pd.Timedelta(hours=24): f["datetime"]]
    if len(w) < 5: continue
    for dt, r in w.iterrows():
        hb = int(round((f["datetime"]-dt)/pd.Timedelta(hours=1)))  # hours before failure
        rows.append((hb, *[ (r[s]-norm[s][0])/norm[s][1] for s in SENS ]))
sig = pd.DataFrame(rows, columns=["hours_before"]+SENS).groupby("hours_before").mean().sort_index(ascending=False)

plt.figure(figsize=(13,4.5))
for s in SENS:
    plt.plot(-sig.index, sig[s], color=PALETTE[s], lw=2.2, marker="o", ms=3, label=s)
plt.axhline(0, color=MUTE, lw=1, ls="--"); plt.axvline(0, color=PALETTE["rose"], lw=1.5, ls=":")
plt.title("Failure signature — average sensor deviation (z-score) in the 24h before failure", fontsize=13)
plt.xlabel("hours relative to failure (0 = failure)"); plt.ylabel("deviation from normal (σ)")
plt.legend(ncol=4); plt.tight_layout(); plt.show()
print("Sensors that swing away from 0 as hour 0 approaches are the ones the classifier keys on.")


In [ ]:
# Do errors spike before failure? error rate in 24h before failure vs a random baseline
err_by_m = {mid: set(g["datetime"]) for mid, g in errs.groupby("machineID")}
def had_error(mid, t, hrs=24):
    s = err_by_m.get(mid, set())
    return any((t-pd.Timedelta(hours=hrs)) <= e <= t for e in s)

pre_fail = np.mean([had_error(f["machineID"], f["datetime"]) for _, f in fails.iterrows()])
rng = np.random.default_rng(0)
samp = tele.sample(2000, random_state=0)
baseline = np.mean([had_error(r["machineID"], r["datetime"]) for _, r in samp.iterrows()])
plt.figure(figsize=(6,3))
plt.bar(["before a failure","random hour"], [pre_fail*100, baseline*100], color=[PALETTE["rose"],PALETTE["slate"]])
plt.title("Chance an error occurred in the prior 24h"); plt.ylabel("%")
for i,v in enumerate([pre_fail*100, baseline*100]): plt.text(i, v+1, f"{v:.0f}%", ha="center", fontweight="bold")
plt.grid(False); plt.tight_layout(); plt.show()
print(f"Errors are ~{pre_fail/max(baseline,1e-9):.1f}x more likely in the 24h before a failure — a strong leading indicator.")


In [ ]:
# How old is a part when it fails? (component age at failure)
from src.pdm.data_loader import load_raw_data
raw = load_raw_data(str(RAW))
from src.pdm.features import add_maintenance_recency_features
base = tele.merge(machines, on="machineID")
rec = add_maintenance_recency_features(base, maint)
agef = []
for _, f in fails.iterrows():
    r = rec[(rec.machineID==f["machineID"]) & (rec.datetime==f["datetime"])]
    if len(r): agef.append(float(r.iloc[0][f"hours_since_{f['failure']}"])/24.0)
plt.figure(figsize=(9,3.2))
plt.hist([a for a in agef if a < 400], bins=40, color=PALETTE["violet"], alpha=0.85)
plt.axvline(np.median(agef), color=INK, ls="--", label=f"median {np.median(agef):.0f}d")
plt.title("Component age at the moment it failed"); plt.xlabel("days since last replacement"); plt.legend()
plt.tight_layout(); plt.show()
print("Failures cluster at higher part ages → 'part age' is the strongest predictor family.")


### 6.1 · The verdict — which data is *main*

Ranked by how much each source drives the prediction (confirmed by the model's SHAP importance):

1. **Component age** (from *maintenance*) — parts fail as they wear; the dominant signal.
2. **Errors** (from *errors*) — ~several× more likely right before a failure.
3. **Sensor trends** (from *telemetry*) — vibration/voltage drift ahead of failure.
4. **Machine type & age** (from *machines*) — context that shifts the baselines.

`failures` never feeds the model — it only defines *truth* for training and validation.


---
## 7 · Data quality & coverage checks ✅

In [ ]:
checks = []
for nm, d in [("telemetry",tele),("errors",errs),("maintenance",maint),("failures",fails),("machines",machines)]:
    checks.append({"table":nm, "rows":len(d), "missing_cells":int(d.isna().sum().sum()),
                   "machines_covered": d["machineID"].nunique() if "machineID" in d else "-",
                   "duplicate_rows": int(d.duplicated().sum())})
display(pd.DataFrame(checks))
# rows per machine (coverage evenness)
rpm = tele.groupby("machineID").size()
plt.figure(figsize=(9,2.6)); plt.hist(rpm, bins=30, color=PALETTE["indigo"])
plt.title(f"Telemetry rows per machine (every machine ≈ {rpm.median():.0f} hrs = full year)"); plt.tight_layout(); plt.show()


---
## 8 · Classification accuracy — Precision, Recall & F1 🎯

The classifier answers *"will this machine fail in the next 12 hours?"*. With ~0.09% positive
rows, **accuracy is misleading** (predict-all-healthy ≈ 99.9%). The honest metrics are:

- **Recall** = of real failures, how many we caught  →  TP / (TP + FN)
- **Precision** = of machines we flagged, how many really failed  →  TP / (TP + FP)
- **F1** = harmonic mean of the two  ·  **AUC-PR** = threshold-free summary for rare events


In [ ]:
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             f1_score, average_precision_score, roc_auc_score)
from src.pdm.scoring import chronological_test_split

REPORTS = ROOT/"outputs"/"reports"
scored = pd.read_parquet(ROOT/"outputs"/"scored.parquet")
y_true, y_prob = chronological_test_split(scored, 0.20)   # same held-out test split as training
THRESH = 0.5
y_pred = (y_prob >= THRESH).astype(int)

prec = precision_score(y_true, y_pred, zero_division=0)
rec  = recall_score(y_true, y_pred, zero_division=0)
f1   = f1_score(y_true, y_pred, zero_division=0)
aucpr = average_precision_score(y_true, y_prob)
aucroc = roc_auc_score(y_true, y_prob)

display(Markdown(f'''
| Metric | Value | Meaning |
|---|---|---|
| **Recall** | **{rec:.3f}** | failures caught |
| **Precision** | **{prec:.3f}** | alerts that are correct |
| **F1** | **{f1:.3f}** | balance of the two |
| **AUC-PR** | **{aucpr:.3f}** | rare-event skill (main metric) |
| **AUC-ROC** | **{aucroc:.3f}** | overall ranking skill |

*Test rows: {len(y_true):,} · actual failures (positives): {int(y_true.sum()):,} · threshold = {THRESH}*
'''))


In [ ]:
# Confusion matrix heatmap
cm = confusion_matrix(y_true, y_pred); tn, fp, fn, tp = cm.ravel()
fig, ax = plt.subplots(figsize=(4.8, 4))
im = ax.imshow(cm, cmap="Blues")
for (i,j), v in np.ndenumerate(cm):
    ax.text(j, i, f"{v:,}", ha="center", va="center", fontsize=13, fontweight="bold",
            color="white" if v > cm.max()/2 else INK)
ax.set_xticks([0,1]); ax.set_xticklabels(["Pred healthy","Pred FAIL"])
ax.set_yticks([0,1]); ax.set_yticklabels(["Actual healthy","Actual FAIL"]); ax.grid(False)
ax.set_title(f"Confusion matrix @ {THRESH}\nTP={tp:,} · FP={fp:,} · FN={fn:,} · TN={tn:,}")
plt.tight_layout(); plt.show()


In [ ]:
# Precision / Recall / F1 as the alert threshold moves
tbl = pd.read_csv(REPORTS/"threshold_table.csv")
plt.figure(figsize=(11, 4))
plt.plot(tbl.threshold, tbl.recall,    color=PALETTE["emerald"], lw=2.4, label="Recall (failures caught)")
plt.plot(tbl.threshold, tbl.precision, color=PALETTE["sky"],     lw=2.4, label="Precision (alerts right)")
plt.plot(tbl.threshold, tbl.f1,        color=PALETTE["amber"],   lw=2.2, ls="--", label="F1 (balance)")
plt.axvline(THRESH, color=PALETTE["slate"], ls=":", label=f"default {THRESH}")
plt.xlabel("alert threshold"); plt.ylabel("score"); plt.ylim(0,1.05)
plt.title("Precision / Recall / F1 vs alert threshold"); plt.legend(); plt.tight_layout(); plt.show()
display(tbl.round(3).head(9))


In [ ]:
# Per-component precision / recall / F1
comp = pd.read_csv(REPORTS/"component_metrics.csv")
display(Markdown("**Per-component metrics**"))
display(comp.round(3))
ax = comp.set_index("component")[["recall","precision","f1"]].plot.bar(
    figsize=(9,3.4), color=[PALETTE["emerald"],PALETTE["sky"],PALETTE["amber"]], width=0.8)
ax.set_title("Precision / Recall / F1 by component"); ax.set_ylim(0,1.05)
plt.xticks(rotation=0); plt.legend(ncol=3, fontsize=8); plt.tight_layout(); plt.show()


**How to read it:** recall is ~1.0 (the model catches virtually every failure); precision
rises as you raise the threshold (fewer, higher-confidence alerts). The **ALERTS** slider in the
app is exactly this threshold — move it left for more recall, right for more precision.


---
## 9 · From raw data → what the model consumes

```
   RAW (what you saw above)                 ENGINEERED (what the model sees)
   ─────────────────────────                ────────────────────────────────
   telemetry ─┐  rolling mean/std/min/max ─▶ volt_mean_24h, vibration_std_12h …
   errors ────┤  rolling counts           ─▶ error4_count_24h …
   maintenance┤  time since replacement    ─▶ hours_since_comp1..4   (top predictor)
   machines ──┘  one-hot + age             ─▶ model_model1..4, age
                                              └─ ~76 features → pruned to top 20
   failures  ─────────────────────────────▶ LABEL: fails within next 12h?  (never a feature)
```

Backward-looking features + forward-looking label = no leakage. That engineered matrix feeds the
**classifier** (12h failure probability, accuracy in §8) and the renewal lives feed the
**Weibull risk score**. See `00_overview_data_and_models.ipynb` for the full modelling walkthrough.
